In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
from reprojection import *
import sunpy.visualization.colormaps as cm

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

dates = []
data = []

for file in files:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        temp = hdul[0].data.copy()

    temp = rebin(temp, 4)
    temp = (temp[1:] + temp[:-1]) / 2
    temp = (temp[:,1:] + temp[:,:-1]) / 2
    data += [np.nan_to_num(temp)]

    date = datetime.fromisoformat(header['T_OBS'][:-4].replace('.', '').replace('_', 'T'))
    dates.append(date)

data = np.array(data)
dates = np.array(dates)

In [3]:
fig, ax = plt.subplots(figsize=(14,8))
ax.imshow(data[17], aspect='auto', origin='lower', cmap='hmimag',
           extent=(0, 360, -1, 1), vmin=-1000, vmax=1000)
ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))
plt.tight_layout()

In [4]:
dsine = 1 / 179.5 #1 / 359.5
lati = np.arange(-1,1 + dsine / 2, dsine)
lati = np.arcsin(lati.clip(-1,1)) * 180 / np.pi
lat = (lati[1:] + lati[:-1]) / 2

dlon = 0.4 # 0.2
loni = np.arange(0,360,dlon)
lon = (loni[1:] + loni[:-1]) / 2

In [22]:
R = 695e8
u = 1000
D = 500e10
lam = 45

dd = 0.1
dt = timedelta(days=dd).total_seconds()

xi = lati * np.pi / 180 * R
vi = u * flow(lati, a=0.5)
wi = (rotation(lat) - rotation(26)) / 24 / 3600 * np.pi / 180 * R

li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = data[0].copy()
Q = []
T = []

for t in np.arange(dates[0], dates[5], timedelta(days=dd)):

    for i in range(y.shape[1]):
        y[:,i] = advect(y[:,i], vi, dt, xi=xi, ai=li)
        y[:,i] = diffuse(y[:,i], D, dt, xi=xi, ai=li)

    for i in range(y.shape[0]):
        y[i] = advect(y[i], wi[i] * np.ones_like(loni), dt, dx=R * dlon * np.pi / 180, boundary='periodic')
        y[i] = diffuse_fft(y[i], D, dt, dx=R * dlon * np.pi / 180)

    y_ = interp1d(data, dates, t)
    y[np.where(np.abs(lat) < lam),:] = y_[np.where(np.abs(lat) < lam),:]

    #Q.append(y.copy())
    T.append(t)

#Q = np.array(Q)
T = np.array(T)

In [23]:
fig, ax = plt.subplots(figsize=(14,8))
ax.imshow(y, aspect='auto', origin='lower', cmap='hmimag',
           extent=(0, 360, -1, 1), vmin=-100, vmax=100)
ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))
plt.tight_layout()

In [24]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 480

view = View(nx, ny, xc, yc, Rsun, -90, 90, 0)
transform = view.to_carrington() + ToSpherical()

grid = view.grid()
grid, _ = transform(grid)
grid = np.array(grid)

grid[0] = np.sin(grid[0] * np.pi / 180) * 180 + 179
grid[1] = grid[1] * 2.5




In [29]:
from show import *

qqq = interp2d(y, *grid)
show_data(qqq, view, vmin=-10, vmax=10, cmap='seismic')

In [26]:
plt.figure(figsize=(10,8))
plt.plot(lat, np.mean(y, axis=-1))


plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

In [52]:
900 / 360

2.5